[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/python_build123d_basics/blob/master/docs/tuto_colab_build/build123d_interactive_exploration.ipynb)

# Caderno de exploração (build123d, modo álgebra): `show()` e `@interactive()`

Mesma exploração do caderno [`interactive_exploration.ipynb`](interactive_exploration.ipynb), trocando apenas a biblioteca CAD: aqui os modelos são construídos com o **build123d em modo álgebra** (`Box(...) - Cylinder(...)`, `Pos(...) * forma`, etc.) em vez do `cadquery.Workplane`. `show()` e `@interactive()` funcionam de forma idêntica — o `cadquery_simpleviewer` não diferencia a biblioteca de origem do objeto. Não faz parte da suíte automatizada de testes — execute as células interativamente e avalie o resultado visualmente.

## Configuração no Google Colab

`build123d` exige um `ipython` mais novo do que o pré-instalado no Colab. A célula abaixo instala a biblioteca (a partir do branch `feature/interactive-sliders`, que ainda não foi publicado no PyPI) e, em seguida, **fixa o `ipython` de volta na versão que o Colab espera, sem reiniciar o runtime** — reiniciar depois desse passo quebra o bootstrap do kernel do Colab. Veja a seção "Google Colab" do README para mais detalhes.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "cadquery-simpleviewer[build123d,interactive]"
         
        check=True,
    )
    # build123d traz um ipython mais novo do que o bootstrap do kernel do
    # Colab tolera. Devolve a versão compatível para o disco — NÃO reinicie
    # o runtime, o kernel atual já está com o ipython funcional carregado.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "ipython==7.34.0", "--no-deps"],
        check=True,
    )
else:
    print("Não é o Colab, pulando a instalação de pacotes.")

In [ ]:
import ipywidgets as widgets
from build123d import Axis, Box, Cylinder, fillet

from cadquery_simpleviewer import show, interactive

## 1. `show()` básico

Verificação simples de que o `show()` renderiza normalmente um objeto do build123d criado em modo álgebra (sem `with BuildPart()`).

In [ ]:
box = Box(5, 3, 2)
show(box)

## 2. Diferença booleana — sombreamento plano vs. suave

O artefato relatado (manchas escuras/moiré) aparecia nas faces curvas criadas por um corte booleano, com `flat_shading=True`. Em modo álgebra, o corte e o friso (fillet) são expressos diretamente com operadores: `Box(...) - Cylinder(...)` e `fillet(arestas, raio)`.

In [ ]:
def caixa_furada_com_friso():
    caixa = Box(20, 20, 20)
    furo = Cylinder(5, 30)
    cortada = caixa - furo
    arestas_verticais = cortada.edges().filter_by(Axis.Z)
    return fillet(arestas_verticais, radius=2)

print("flat_shading=True (padrão antigo) — espera-se aspecto facetado/moiré no furo/friso")
show(caixa_furada_com_friso(), flat_shading=True)

In [ ]:
print("flat_shading=False (novo padrão) — espera-se superfícies curvas suaves")
show(caixa_furada_com_friso(), flat_shading=False)

In [ ]:
print("angular_tolerance mais fina — malha mais densa nas faces curvas")
show(caixa_furada_com_friso(), angular_tolerance=0.02)

## 3. `@interactive()` — sliders básicos

Ao decorar a função, os sliders e o modelo renderizado aparecem imediatamente. Arraste largura/altura e solte para disparar a reconstrução (padrão `continuous_update=False`).

In [ ]:
@interactive(width=(1, 10, 0.5, 5), height=(1, 8, 0.5, 3))
def modelo_caixa(width, height):
    return Box(width, height, 2)

A própria função decorada é devolvida sem alterações — continua chamável diretamente, sem o efeito colateral do widget:

In [ ]:
modelo_caixa(4, 4).volume

## 4. `@interactive()` com `show_kwargs`

Opções de exibição (cores, plano de base, etc.) são passadas em um dicionário `show_kwargs` separado, para nunca colidirem com o nome de um parâmetro do modelo.

In [ ]:
@interactive(
    width=(1, 10, 0.5, 5),
    height=(1, 8, 0.5, 3),
    show_kwargs=dict(colors=["steelblue"], z=0, plane_color="gainsboro"),
)
def caixa_com_plano(width, height):
    return Box(width, height, 2)

## 5. `@interactive()` com um controle `ipywidgets` explícito

Qualquer controle pode ser um widget já pronto, em vez de uma tupla `(min, max, step, default)` — útil para `Dropdown`, `Checkbox`, etc.

In [ ]:
@interactive(forma=widgets.Dropdown(options=["caixa", "cilindro"], value="caixa"))
def modelo_por_forma(forma):
    if forma == "caixa":
        return Box(4, 4, 4)
    return Cylinder(2, 4)

## 6. `@interactive()` com `continuous_update=True`

Reconstrução em tempo real a cada movimento do slider, em vez de apenas ao soltar — funciona bem aqui pois a caixa é barata de reconstruir/tesselar, mas pode travar em geometrias mais pesadas (ex.: a caixa furada com friso da seção 2).

In [ ]:
@interactive(width=(1, 10, 0.5, 5), continuous_update=True)
def caixa_ao_vivo(width):
    return Box(width, 3, 2)

## 7. `@interactive()` controlando uma diferença booleana

Combina as seções 2 e 3 — raio do furo controlado por slider em um corte booleano, verificando se a correção do sombreamento suave se mantém sob mudanças de parâmetro em tempo real.

In [ ]:
@interactive(raio_furo=(1, 9, 0.5, 5), raio_friso=(0.5, 4, 0.5, 2))
def modelo_com_corte(raio_furo, raio_friso):
    caixa = Box(20, 20, 20)
    furo = Cylinder(raio_furo, 30)
    cortada = caixa - furo
    arestas_verticais = cortada.edges().filter_by(Axis.Z)
    return fillet(arestas_verticais, radius=raio_friso)

## 8. `@interactive()` com retorno de dicionário — `show_kwargs` por quadro

Em vez de devolver só o(s) objeto(s), a função decorada pode devolver um `dict` com uma chave `"objects"` (o(s) objeto(s) a renderizar) e quaisquer outras chaves aceitas por `show()`/`_build_figure()` (`colors`, `opacity`, `z`, ...), que sobrescrevem `show_kwargs` **apenas naquele quadro** — útil, por exemplo, para sinalizar visualmente uma geometria inválida. Arraste o slider: abaixo de 5 a caixa fica azul (`steelblue`, o `show_kwargs` padrão); a partir de 5 fica vermelha (`indianred`, sobrescrita pelo dict retornado).

In [ ]:
@interactive(radius=(1, 9, 1, 5), show_kwargs=dict(colors=["steelblue"]))
def modelo_com_alerta(radius):
    box = Box(10, 10, 2)
    if radius >= 5:
        return {"objects": box, "colors": ["indianred"]}
    return box